# Semantic Search with ChromaDB, a Vector Database

---


### What is ChromaDB?
ChromaDB is an open-source vector database designed to store and manage high-dimensional vector embeddings.

### Why use ChromaDB for Semantic Search?
- Efficient Storage and Retrieval: ChromaDB is optimized for storing and retrieving large volumes of vector data quickly.
- Scalability: It can handle large datasets, making it suitable for applications with extensive
    document collections.

### What we will do in this notebook:
1) Convert text → embeddings (vectors)
2) Store embeddings in a vector database
3) Query by semantic meaning, not by exact keywords

That’s the core of Retrieval-Augmented Generation (RAG) used in systems like ChatGPT with external knowledge.

---

### 1) Initialize the embedding model and Create Chroma Client

In [7]:
from sentence_transformers import SentenceTransformer
import chromadb

# Initialize the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a Chroma client (in-memory DB)
chroma_client = chromadb.Client()

# Docs already exists, so we just get it
collection = chroma_client.get_collection("docs")
# If the collection did not exist, you would create it like this:
#collection = chroma_client.create_collection("docs")


---
### 2) Add some example documents

- First we create the documents with text
- Then we convert the document content into vectors (embeddings) using the embedding model
   * An embedding is a numerical representation of text that captures its semantic meaning, and a model is used to generate these embeddings.
- Finally we add the documents and their embeddings to the ChromaDB collection

In [8]:
documents = [
    "The cat sat on the mat.",
    "Dogs are loyal animals.",
    "The stock market is volatile today.",
    "Artificial Intelligence is transforming the world.",
    "I love programming in Python."
]

# Convert documents into vectors (embeddings)
embeddings = model.encode(documents).tolist()

# Store them in the vector database
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc{i}" for i in range(len(documents))]
)


---
### 4) Perform a similarity search


In [9]:
query = "machine learning and AI"
query_embedding = model.encode([query]).tolist()

# Search for the most similar documents
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

print(results)


{'ids': [['doc3', 'doc4']], 'embeddings': None, 'documents': [['Artificial Intelligence is transforming the world.', 'I love programming in Python.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.9398390054702759, 1.6456501483917236]]}


### Optional: Loop results for better formatting

In [22]:
### Optional: Loop results for better formatting
result = {'ids': [['doc3', 'doc4']], 'embeddings': None, 'documents': [['Artificial Intelligence is transforming the world.', 'I love programming in Python.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.9398390054702759, 1.6456501483917236]]}

print("\n🔍 Search Results for query:", query)

for doc_id, doc_text in zip(result["ids"][0], result["documents"][0]):
    print(f"\n📄 File: {doc_id}\n{doc_text[:300]}...")  # show first 300 chars


🔍 Search Results for query: What is artificial intelligence?

📄 File: doc3
Artificial Intelligence is transforming the world....

📄 File: doc4
I love programming in Python....
